## Utility Functions

The function `range` is *overloaded*:
 * The function `range(start, stop)` creates the list `[start, ..., stop]`.
 * The function `range(stop)` creates the list `[0, ..., stop]`.

In [ ]:
function range(stop: number): number[];

function range(start: number, stop: number): number[];

function range(startOrStop: number, stop?: number): number[] {
    const start = stop === undefined ? 0 : startOrStop;
    const end   = stop === undefined ? startOrStop : stop;
    
    return Array.from({ length: end - start + 1 }, (_, index) => start + index);
}

In [ ]:
range(3)

In [ ]:
range(2, 5)

# The Knight's Tour

This notebook computes a solution to the [knight's tour](https://en.wikipedia.org/wiki/Knight%27s_tour) using the constraint solver `Z3`.  

In [ ]:
import { init, Bool, BitVec } from 'z3-solver';
const { Context } = await init();
const Z3 = Context("main");

Given an integer from the set $\{0, 1, \cdots, 63\}$, the function `row(i)` computes the name of the variable that specifies the *row* of the knight after its $i^{\textrm{th}}$ move.

In [ ]:
function row(i: number): string {
    return `R${i}`;
}

In [ ]:
row(1)

Given an integer from the set $\{0, 1, \cdots, 63\}$, the function `col(i)` computes the name of the variable that specifies the *column* of the knight after its $i^{\textrm{th}}$ move.

In [ ]:
function col(i: number): string {
    return `C${i}`;
}

The function `toBitVec(x)` turns the number `x` into a *bit vector* of length 4.

In [ ]:
const toBitVec = (x: number): BitVec => Z3.BitVec.val(x, 4);

The function `toBitVar(x)` turns a string into a variable that has a type of bit vector of length 4.

In [ ]:
const toBitVar = (x: string): BitVec => Z3.BitVec.const(x, 4);

The function `isKnightMove(row, col, rowX, colX)` takes four arguments:
* `row` is a `Z3` variable that specifies the row of the position of the knight before the move.
* `col` is a `Z3` variable that specifies the column of the position of the knight before the move.
* `rowX` is a `Z3` variable that specifies the row of the position of the knight after the move.
* `colX` is a `Z3` variable that specifies the column of the position of the knight after the move.

It returns a formula that specifies that the specified position represents a legal move for a knight.

In order to form the *conjunction* of two formulas we use the method `Z3.And`, 
while the *disjunction* is build with the function `Z3.Or`.  
Note that these functions can be called with any number of arguments.

The figure below shows the moves of a knight:  The knight on `e4` can jump to all red squares.
<img src="knight-moves.png" width="50%">

In [ ]:
function isKnightMove(row: BitVec, col: BitVec, rowX: BitVec, colX: BitVec): Bool {
    const deltaSet = [[1, 2],  [1, -2], [2, 1], [2, -1], [-1, 2], [-1, -2], [-2, 1], [-2, -1]];
    const formulas = deltaSet.map(([deltaRow, deltaCol]) =>                         
        Z3.And(rowX.eq(row.add(toBitVec(deltaRow))),      // rowX == row + deltaRow,
               colX.eq(col.add(toBitVec(deltaCol))) ));   // colX == col + deltaCol 
    return Z3.Or(...formulas);
}

The function `allDifferent` takes two arguments:
* `Rows` is a list of `Z3` variables. The variable `Rows[i]` specifies the row of the position of the knight after the $i^{\textrm{th}}$ move.
* `Cols` is a list of `Z3` variables. The variable `Cols[i]` specifies the column of the position of the knight after the $i^{\textrm{th}}$ move.

The function returns a set of formulas stating that for $i \not= j$ the positions after the $i^{\textrm{th}}$ move
differs from the position after the $j^{\textrm{th}}$ move.

In [ ]:
function allDifferent(Rows: BitVec[], Cols: BitVec[]): Bool[] {
    return range(62).flatMap(i => 
        range(i + 1, 63).map(j => Z3.Or(Rows[i].neq(Rows[j]), Cols[i].neq(Cols[j])))
    );  // [Ri, Ci] != [Rj, Cj]  f.a. i < j
}

The function `allConstraints` takes two arguments:
* `Rows` is a list of `Z3` variables. The variable `Rows[i]` specifies the row of the position of the knight after the $i^{\textrm{th}}$ move.
* `Cols` is a list of `Z3` variables. The variable `Cols[i]` specifies the column of the position of the knight after the $i^{\textrm{th}}$ move.

`allConstraints` returns a set containing all constraints of the problem.

In [ ]:
function allConstraints(Rows: BitVec[], Cols: BitVec[]): Bool[] {
    const zero  = toBitVec(0);
    const eight = toBitVec(8);
    return [...allDifferent(Rows, Cols), 
            Rows[0].eq(zero), Cols[0].eq(zero),
            ...range(62).map(i => isKnightMove(Rows[i], Cols[i], Rows[i+1], Cols[i+1])),
            ...range(63).flatMap(i => [Rows[i].ult(eight), Cols[i].ult(eight)])
           ];
}

In [ ]:
function parseVal(expr: BitVec): number {
    return parseInt(expr.toString().substring(2), 16);
}

The function `solve()` computes a solution of the knight's problem and returns this solution.

In [ ]:
async function solve(): Promise<[number, number][]> {
    const Rows: BitVec[] = range(63).map(i => toBitVar(row(i)));
    const Cols: BitVec[] = range(63).map(i => toBitVar(col(i)));;
    const solver = new Z3.Solver();
    solver.add(...allConstraints(Rows, Cols));
    const check = await solver.check();      
    if (check == 'sat') {
        const model = solver.model();
        const solution: [number, number][] = range(63).map(i => 
            [parseVal(model.eval(Rows[i])), parseVal(model.eval(Cols[i]))]);
        return solution;
    } else if (check == 'unsat') {
        console.log('The problem is not solvable.');
    } else {
        console.log('Z3 cannot determine whether the problem is solvable.');
    }
}

Unfortunately, the execution time of the following cell varies greatly between
different runs.  Sometimes the cell runs in less one minute and 28 seconds, sometimes 
it might take 30 minutes.

In [ ]:
console.time("Solver Duration");
const Solution = await solve();
console.timeEnd("Solver Duration");
Solution

The function `createBoard(Solution)` returns a matrix `Board` of size $8\times 8$.
The following holds:
$$ \texttt{Board}[\texttt{R}i][\texttt{C}i] = i $$
Therefore, if `Board[r][c] == i`, then at the beginning of the $i^{\textrm{th}}$ move the knight is located in row `r` and column `c`. 

In [ ]:
function createBoard(solution: [number, number][]) {
    const board: number[][] = Array.from({ length: 8 }, () => Array(8).fill(0));
    solution.forEach(([r, c], i) => { board[r][c] = i; });    
    return board;
}

In [ ]:
createBoard(Solution)

The function `printBoard` prints the given `Board`.

In [ ]:
function printBoard(board: number[][]) {
    const n = board.length;
    if (n === 0) return;
    const width = board
        .flat()
        .reduce((max, element) => Math.max(max, element.toString().length), 0);
    const createLine = (left: string, mid: string, right: string, fill: string) => {
        let line = left;
        for (let i = 0; i < n - 1; i++) {
            line += fill.repeat(width + 2) + mid;
        }
        line += fill.repeat(width + 2) + right;
        return line;
    };
    const topLine = createLine('╔', '╦', '╗', '═');
    const midLine = createLine('╠', '╬', '╣', '═');
    const botLine = createLine('╚', '╩', '╝', '═');
    console.log(topLine);
    board.forEach((row, i) => {
        let line = '\u2551';
        for (const element of row) {
            const s = element.toString().padStart(width, ' ');
            line += ` ${s} ║`;
        }
        console.log(line);
        if (i < n - 1) {
            console.log(midLine);
        }
    });
    console.log(botLine);
}

In [ ]:
const board = createBoard(Solution);

In [ ]:
printBoard(board)

# Visualization

The function `showSolution` displays the given solution on a chessboard.
The solution `Board` is represented as a list of lists.  We have `Board[row][col] == k` if the $k^\textrm{th}$ move leads the knight to the position `(row, col)`.

In [ ]:
import * as tslab from "tslab";

function showSolution(board: number[][], width: string = "50%"): void {
    const n = board.length;
    const cellSize = 50;
    const totalSize = n * cellSize;
    const path: [number, number][] = new Array(n * n);
    
    for (let r = 0; r < n; r++) {
        for (let c = 0; c < n; c++) {
            const k = board[r][c];
            if (k >= 0 && k < n * n) {
                path[k] = [r, c];
            }
        }
    }
    
    const uniqueId = Math.random().toString(36).substring(2, 11);
    
    let html = `<div id="tour-container-${uniqueId}">`;
    
    html += `<button id="btn-${uniqueId}" onclick="playKnightTour_${uniqueId}()" style="padding: 10px 15px; margin-bottom: 10px; cursor: pointer; background: #4CAF50; color: white; border: none; border-radius: 4px; font-size: 14px;">▶ Play Epic Tour</button><br>`;
    
    html += `<svg width="${width}" viewBox="0 0 ${totalSize} ${totalSize}" xmlns="http://www.w3.org/2000/svg" style="font-family: sans-serif;">`;
    
    // Draw board
    for (let r = 0; r < n; r++) {
        for (let c = 0; c < n; c++) {
            const isBlack = (r + c) % 2 !== 0;
            const color = isBlack ? '#b58863' : '#f0d9b5';
            const x = c * cellSize;
            const y = r * cellSize;         
            html += `<rect x="${x}" y="${y}" width="${cellSize}" height="${cellSize}" fill="${color}" />`;
            html += `<text x="${x + cellSize/2}" y="${y + cellSize/2 + 5}" text-anchor="middle" font-size="14" fill="${isBlack ? '#f0d9b5' : '#b58863'}">${board[r][c]}</text>`;
        }
    }
    
    // Start position
    if (path[0]) {
        const [startR, startC] = path[0];
        html += `<circle cx="${startC * cellSize + cellSize/2}" cy="${startR * cellSize + cellSize/2}" r="8" fill="darkblue" />`;
    }
    
    // Path lines and dots
    for (let k = 0; k < n * n - 1; k++) {
        const p1 = path[k];
        const p2 = path[k+1];
        if (p1 && p2) {
            const x1 = p1[1] * cellSize + cellSize/2;
            const y1 = p1[0] * cellSize + cellSize/2;
            const x2 = p2[1] * cellSize + cellSize/2;
            const y2 = p2[0] * cellSize + cellSize/2;
            
            html += `<line id="line-${uniqueId}-${k}" x1="${x1}" y1="${y1}" x2="${x2}" y2="${y2}" stroke="darkblue" stroke-width="3" stroke-opacity="0.6" opacity="0"></line>`;
            html += `<circle id="circle-${uniqueId}-${k}" cx="${x2}" cy="${y2}" r="3" fill="darkblue" opacity="0"></circle>`;
        }
    }
    
    html += `</svg>`;
    
    // Animation and Audio Logic
    html += `
    <script>
        let isPlaying_${uniqueId} = false;

        function playKnightTour_${uniqueId}() {
            if (isPlaying_${uniqueId}) return;
            isPlaying_${uniqueId} = true;
            
            const btn = document.getElementById('btn-${uniqueId}');
            btn.style.background = '#888';
            btn.innerText = 'Running...';

            const maxSteps = ${n * n - 1};
            const delay = 500; // Half speed (was 250ms)
            
            const AudioContext = window.AudioContext || window.webkitAudioContext;
            const audioCtx = new AudioContext();
            if (audioCtx.state === 'suspended') audioCtx.resume();
            
            // --- 1. SETUP REVERB ---
            const convolver = audioCtx.createConvolver();
            const rate = audioCtx.sampleRate;
            const length = rate * 2.0; // 2 seconds of reverb decay
            const impulse = audioCtx.createBuffer(2, length, rate);
            for (let i = 0; i < length; i++) {
                const decay = Math.exp(-i / (rate * 0.3)); // Reverb tail
                impulse.getChannelData(0)[i] = (Math.random() * 2 - 1) * decay;
                impulse.getChannelData(1)[i] = (Math.random() * 2 - 1) * decay;
            }
            convolver.buffer = impulse;
            convolver.connect(audioCtx.destination);

            // --- 2. SOUND DESIGN FUNCTIONS ---
            function playNeigh() {
                const osc = audioCtx.createOscillator();
                const lfo = audioCtx.createOscillator();
                const tremolo = audioCtx.createGain();
                const mainGain = audioCtx.createGain();
                
                // The base sound: A descending sawtooth
                osc.type = 'sawtooth';
                osc.frequency.setValueAtTime(900, audioCtx.currentTime);
                osc.frequency.exponentialRampToValueAtTime(300, audioCtx.currentTime + 1.2);
                
                // Tremolo effect to simulate the "chatter" of a whinny
                lfo.type = 'sine';
                lfo.frequency.value = 18; 
                tremolo.gain.value = 0.8;
                
                mainGain.gain.setValueAtTime(0, audioCtx.currentTime);
                mainGain.gain.linearRampToValueAtTime(0.5, audioCtx.currentTime + 0.1);
                mainGain.gain.setValueAtTime(0.5, audioCtx.currentTime + 0.8);
                mainGain.gain.exponentialRampToValueAtTime(0.01, audioCtx.currentTime + 1.2);

                lfo.connect(tremolo.gain);
                osc.connect(tremolo);
                tremolo.connect(mainGain);
                
                // Route to both direct output and reverb
                mainGain.connect(audioCtx.destination);
                mainGain.connect(convolver);
                
                lfo.start(); osc.start();
                lfo.stop(audioCtx.currentTime + 1.3); osc.stop(audioCtx.currentTime + 1.3);
            }

            function playMoveSound() {
                const osc = audioCtx.createOscillator();
                const gainNode = audioCtx.createGain();
                
                osc.type = 'sine';
                osc.frequency.setValueAtTime(300, audioCtx.currentTime);
                osc.frequency.exponentialRampToValueAtTime(500, audioCtx.currentTime + 0.05);
                
                gainNode.gain.setValueAtTime(0.4, audioCtx.currentTime);
                gainNode.gain.exponentialRampToValueAtTime(0.001, audioCtx.currentTime + 0.05);
                
                osc.connect(gainNode);
                
                // Route to direct output AND reverb to create the echo
                gainNode.connect(audioCtx.destination);
                gainNode.connect(convolver);
                
                osc.start();
                osc.stop(audioCtx.currentTime + 0.05);
            }

            function playVictorySound() {
                const notes = [261.63, 329.63, 392.00, 523.25]; // C4, E4, G4, C5 Fanfare
                const startTime = audioCtx.currentTime;
                
                notes.forEach((freq, index) => {
                    const osc = audioCtx.createOscillator();
                    const gain = audioCtx.createGain();
                    osc.type = 'triangle';
                    osc.frequency.value = freq;
                    
                    const time = startTime + index * 0.15;
                    const duration = index === notes.length - 1 ? 2.0 : 0.15;
                    
                    gain.gain.setValueAtTime(0, time);
                    gain.gain.linearRampToValueAtTime(0.3, time + 0.05);
                    gain.gain.exponentialRampToValueAtTime(0.01, time + duration);
                    
                    osc.connect(gain);
                    gain.connect(audioCtx.destination);
                    gain.connect(convolver);
                    
                    osc.start(time);
                    osc.stop(time + duration);
                });
            }

            // --- 3. ANIMATION SEQUENCE ---
            // Reset visuals
            for (let i = 0; i < maxSteps; i++) {
                const line = document.getElementById('line-${uniqueId}-' + i);
                const circle = document.getElementById('circle-${uniqueId}-' + i);
                if (line) line.setAttribute('opacity', '0');
                if (circle) circle.setAttribute('opacity', '0');
            }

            // Play the opening neigh, then wait 1.5 seconds before starting the tour
            playNeigh();
            
            setTimeout(() => {
                let currentStep = 0;
                const intervalId = setInterval(() => {
                    if (currentStep >= maxSteps) {
                        clearInterval(intervalId);
                        playVictorySound();
                        btn.style.background = '#4CAF50';
                        btn.innerText = '▶ Replay Tour';
                        isPlaying_${uniqueId} = false;
                        return;
                    }
                    
                    // Reveal SVG paths
                    const line = document.getElementById('line-${uniqueId}-' + currentStep);
                    const circle = document.getElementById('circle-${uniqueId}-' + currentStep);
                    if (line) line.setAttribute('opacity', '1');
                    if (circle) circle.setAttribute('opacity', '1');
                    
                    playMoveSound();
                    currentStep++;
                    
                }, delay);
            }, 1500); // 1.5 second pause for the horse neigh
        }
    </script>
    </div>`;
    
    tslab.display.html(html);
}

In [ ]:
showSolution(board)